# 軸1: BPR の次元・正則化で 7 本の提出ファイルを作成

**参照**: `docs/08_COLLABORATIVE_FILTERING.md` の **§5.5 協調フィルタをさらに効果的に使うための実装の余地** の軸1。

現時点の最高は **BPR factors=64**（`submission_2hop_bpr64_only.csv` の 0.76479）。ここから「次元をさらに増やす」「正則化・イテレーションを変える」で 7 本の提出を生成する。

| # | 実験名 | factors | bpr_kwargs | 狙い |
|---|--------|---------|------------|------|
| 1 | axis1_bpr128 | 128 | なし | 次元 128 で表現力アップ |
| 2 | axis1_bpr256 | 256 | なし | 次元 256 でさらに伸ばす |
| 3 | axis1_bpr128_reg_high | 128 | regularization=0.1 | 過学習抑制 |
| 4 | axis1_bpr128_reg_low | 128 | regularization=0.001 | 弱い正則化 |
| 5 | axis1_bpr128_iter200 | 128 | iterations=200 | 収束を良くする |
| 6 | axis1_bpr256_reg_high | 256 | regularization=0.1 | 256 + 過学習抑制 |
| 7 | axis1_bpr64_reg_high | 64 | regularization=0.1 | 現ベスト次元で正則化 |

提出先: `outputs/submissions/submission_2hop_{実験名}.csv`

## 1. セットアップ

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from lib.improvement_candidates import get_setup
from lib.two_hop import run_experiment

ctx = get_setup(seed=42, output_dir="outputs", use_best_pipeline=True)

print(f"Train: {len(ctx.train):,}, Test: {len(ctx.test):,}, Features: {len(ctx.features)}")
print(f"提出先: {ctx.submissions_dir}")

Train: 653,507, Test: 40,716, Features: 55
提出先: outputs/submissions


## 2. 軸1 実験 7 本を実行（BPR のみ・2-hop なし）

In [2]:
# 軸1: (実験名, bpr_factors, bpr_kwargs)
AXIS1_EXPERIMENTS = [
    ("axis1_bpr128", 128, None),
    ("axis1_bpr256", 256, None),
    ("axis1_bpr128_reg_high", 128, {"regularization": 0.1}),
    ("axis1_bpr128_reg_low", 128, {"regularization": 0.001}),
    ("axis1_bpr128_iter200", 128, {"iterations": 200}),
    ("axis1_bpr256_reg_high", 256, {"regularization": 0.1}),
    ("axis1_bpr64_reg_high", 64, {"regularization": 0.1}),
]

results = []
for i, (name, factors, bpr_kwargs) in enumerate(AXIS1_EXPERIMENTS, start=1):
    print(f"[{i}/7] {name} (factors={factors}, bpr_kwargs={bpr_kwargs}) ...")
    out = run_experiment(
        ctx,
        experiment_name=name,
        bpr_factors=factors,
        bpr_kwargs=bpr_kwargs,
    )
    results.append({
        "name": name,
        "path": out.get("path"),
        "ok": out.get("ok"),
        "message": out.get("message", ""),
    })
    print(f"  → {out.get('path', '')}  (OK={out.get('ok')}, {out.get('message', '')})")
n_ok = sum(1 for r in results if r.get("ok"))
print(f"\nDone. 検証: {n_ok}/7 が OK（提出形式: ID, target, 行数=test と一致, target は [0,1]）. 7 本が outputs/submissions に保存されています。")

[1/7] axis1_bpr128 (factors=128, bpr_kwargs=None) ...


  0%|          | 0/100 [00:00<?, ?it/s]

  → outputs/submissions/submission_2hop_axis1_bpr128.csv  (OK=True, OK)
[2/7] axis1_bpr256 (factors=256, bpr_kwargs=None) ...


  0%|          | 0/100 [00:00<?, ?it/s]

  → outputs/submissions/submission_2hop_axis1_bpr256.csv  (OK=True, OK)
[3/7] axis1_bpr128_reg_high (factors=128, bpr_kwargs={'regularization': 0.1}) ...


...

  → outputs/submissions/submission_2hop_axis1_bpr128_iter200.csv  (OK=True, OK)
[6/7] axis1_bpr256_reg_high (factors=256, bpr_kwargs={'regularization': 0.1}) ...


  0%|          | 0/100 [00:00<?, ?it/s]

  → outputs/submissions/submission_2hop_axis1_bpr256_reg_high.csv  (OK=True, OK)
[7/7] axis1_bpr64_reg_high (factors=64, bpr_kwargs={'regularization': 0.1}) ...


  0%|          | 0/100 [00:00<?, ?it/s]

  → outputs/submissions/submission_2hop_axis1_bpr64_reg_high.csv  (OK=True, OK)

Done. 検証: 7/7 が OK（提出形式: ID, target, 行数=test と一致, target は [0,1]）. 7 本が outputs/submissions に保存されています。


## 3. 保存されたファイル一覧

In [3]:
import pandas as pd

summary = pd.DataFrame(results)
for _, r in summary.iterrows():
    p = Path(r["path"]) if r.get("path") else ctx.submissions_dir / f"submission_2hop_{r['name']}.csv"
    size_mb = p.stat().st_size / (1024 * 1024) if p.exists() else 0
    ok_str = "OK" if r.get("ok") else r.get("message", "?")
    print(f"  {p.name}  ({size_mb:.2f} MB)  verify={ok_str}")
# 1本だけ中身を確認: 列は ID, target で行数は test と一致していること
if results:
    p0 = Path(results[0]["path"]) if results[0].get("path") else ctx.submissions_dir / f"submission_2hop_{results[0]['name']}.csv"
    if p0.exists():
        sub = pd.read_csv(p0)
        print(f"\n先頭1本の形式: columns={list(sub.columns)}, rows={len(sub)} (test={len(ctx.test)})")
summary

  submission_2hop_axis1_bpr128.csv  (0.00 MB)  verify=OK
  submission_2hop_axis1_bpr256.csv  (0.00 MB)  verify=OK
  submission_2hop_axis1_bpr128_reg_high.csv  (0.00 MB)  verify=OK
  submission_2hop_axis1_bpr128_reg_low.csv  (0.00 MB)  verify=OK
  submission_2hop_axis1_bpr128_iter200.csv  (0.00 MB)  verify=OK
  submission_2hop_axis1_bpr256_reg_high.csv  (0.00 MB)  verify=OK
  submission_2hop_axis1_bpr64_reg_high.csv  (1.01 MB)  verify=OK


,name,path,ok,message
0,axis1_bpr128,outputs/submissions/submission_2hop_axis1_bpr1...,True,OK
1,axis1_bpr256,outputs/submissions/submission_2hop_axis1_bpr2...,True,OK
2,axis1_bpr128_reg_high,outputs/submissions/submission_2hop_axis1_bpr1...,True,OK
3,axis1_bpr128_reg_low,outputs/submissions/submission_2hop_axis1_bpr1...,True,OK
4,axis1_bpr128_iter200,outputs/submissions/submission_2hop_axis1_bpr1...,True,OK
5,axis1_bpr256_reg_high,outputs/submissions/submission_2hop_axis1_bpr2...,True,OK
6,axis1_bpr64_reg_high,outputs/submissions/submission_2hop_axis1_bpr6...,True,OK
